In [1]:
#%pip install torch torchvision matplotlib numpy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# 1. Device Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Load Fashion-MNIST Dataset
transform = transforms.ToTensor()

train_dataset = datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

train_loader = torch.utils.data.DataLoader(
    dataset=train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    dataset=test_dataset,
    batch_size=1000,
    shuffle=False
)

# 3. Define Feedforward Neural Network (MLP)
class FeedforwardNN(nn.Module):

    def __init__(self):
        super(FeedforwardNN, self).__init__()

        self.fc1 = nn.Linear(28 * 28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 28 * 28)      # Flatten the image
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


# 4. Initialize Model, Loss, and Optimizer
model = FeedforwardNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

# 5. Train the Model
epochs = 10

train_losses = []
test_losses = []
accuracy_list = []

for epoch in range(epochs):

    model.train()

    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    train_losses.append(running_loss / len(train_loader))

    # Evaluation
    model.eval()

    correct = 0
    total = 0
    test_loss = 0.0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            test_loss += criterion(outputs, labels).item()

            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    test_losses.append(test_loss / len(test_loader))

    accuracy = 100 * correct / total

    accuracy_list.append(accuracy)

    print(
        f"Epoch [{epoch+1}/{epochs}], "
        f"Train Loss: {train_losses[-1]:.4f}, "
        f"Test Loss: {test_losses[-1]:.4f}, "
        f"Accuracy: {accuracy:.2f}%"
    )

# 6. Plot Accuracy and Loss Curves
plt.figure(figsize=(12, 5))

# Loss Curve
plt.subplot(1, 2, 1)

plt.plot(train_losses, label='Train Loss')
plt.plot(test_losses, label='Test Loss')

plt.title('Loss Curves')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Accuracy Curve
plt.subplot(1, 2, 2)

plt.plot(accuracy_list, label='Test Accuracy', color='green')

plt.title('Accuracy Curve')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.legend()

plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'torchvision'